In [1]:
import pandas as pd, numpy as np
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"

In [2]:
r = pd.read_csv(RAW / "rainfall_2022_2026.csv", parse_dates=["time"]).set_index("time")
r = r.asfreq("h")                                   # force a strict hourly index
assert r["precipitation"].isna().sum() == 0, "missing hours in rainfall data"

rain = pd.DataFrame(index=r.index)
rain["rain_1h"] = r["precipitation"]
for h in [3, 6, 24, 72]:
    rain[f"rain_{h}h"] = r["precipitation"].rolling(h, min_periods=h).sum()

rain.to_csv(PROC / "rain_features.csv")
print("rain_features:", rain.shape)

rain_features: (41880, 5)


In [3]:
rain = pd.read_csv(PROC / "rain_features.csv", index_col=0, parse_dates=True)
daily = rain["rain_1h"].resample("D").sum()
peak_hr = rain["rain_1h"].groupby(rain.index.date).idxmax()

top = daily.sort_values(ascending=False).head(15).round(1).to_frame("day_total_mm")
top["peak_hour"] = [peak_hr[d.date()].strftime("%H:%M") for d in top.index]
print(top.sort_index())

            day_total_mm peak_hour
time                              
2022-06-30          47.7     05:00
2022-09-14          40.2     03:00
2022-09-15          82.5     21:00
2023-07-01          39.4     18:00
2024-07-01          58.6     23:00
2024-07-02          80.0     07:00
2024-07-05          44.2     17:00
2024-08-09          40.7     13:00
2024-09-11          57.7     15:00
2024-09-12          57.9     20:00
2025-06-30          38.7     14:00
2025-08-03          40.8     07:00
2025-08-04          66.8     03:00
2026-09-25          93.8     20:00
2026-09-26          41.1     03:00


In [4]:
e = pd.read_csv(RAW / "lucknow_grid_elevation.csv")
coords, elev = e[["lat", "lon"]].to_numpy(), e["elevation"].to_numpy()
dist = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(-1))

mask = (dist > 0) & (dist <= 0.025)          # neighbours within ~2.5 km, self excluded
e["relative_elev"] = elev - (mask * elev[None, :]).sum(1) / mask.sum(1)

e.to_csv(PROC / "grid_terrain.csv", index=False)
print(e["relative_elev"].describe().round(2))

count    483.00
mean       0.04
std        2.82
min      -12.15
25%       -1.57
50%       -0.25
75%        1.70
max       12.93
Name: relative_elev, dtype: float64


In [5]:
rain = pd.read_csv(PROC / "rain_features.csv", index_col=0, parse_dates=True)
grid = pd.read_csv(PROC / "grid_terrain.csv")
labels = pd.read_csv(ROOT / "data" / "labels" / "waterlogging_reports.csv", parse_dates=["date"])
print("labels loaded:", len(labels))

labels["label"] = (labels["severity"].str.lower() != "none").astype(int)

rows = []
for _, r in labels.iterrows():
    day = rain[rain.index.strftime("%Y-%m-%d") == r["date"].strftime("%Y-%m-%d")]
    if day.empty:
        print("No rain data for", r["date"].date()); continue
    hour = day["rain_1h"].idxmax()

    d = (grid["lat"] - r["lat"]) ** 2 + (grid["lon"] - r["lon"]) ** 2
    if np.sqrt(d.min()) > 0.01:
        print("Skipping (too far from grid):", r["location_name"]); continue
    t = grid.loc[d.idxmin()]

    rows.append({
        "location_name": r["location_name"], "event_hour": hour, "label": r["label"],
        **rain.loc[hour, ["rain_1h", "rain_3h", "rain_6h", "rain_24h", "rain_72h"]].to_dict(),
        "elevation": t["elevation"], "relative_elev": t["relative_elev"],
    })

table = pd.DataFrame(rows)
table.to_csv(PROC / "training_table.csv", index=False)
print(table.round({"rain_1h": 1, "rain_3h": 1, "rain_6h": 1, "rain_24h": 1, "rain_72h": 1,
                   "elevation": 0, "relative_elev": 1}).sort_values(["event_hour", "location_name"]))
print("\nLabel counts:\n", table["label"].value_counts())

labels loaded: 12
Skipping (too far from grid): Sarojini Nagar
   location_name          event_hour  label  rain_1h  rain_3h  rain_6h  \
7       Ashiyana 2024-07-07 06:00:00      1      2.5      2.6      2.8   
10  Faizullaganj 2024-07-07 06:00:00      1      2.5      2.6      2.8   
8    Gomti Nagar 2024-07-07 06:00:00      1      2.5      2.6      2.8   
9   Indira Nagar 2024-07-07 06:00:00      1      2.5      2.6      2.8   
6       Alambagh 2026-09-25 20:00:00      1     13.0     33.2     50.1   
2        Aliganj 2026-09-25 20:00:00      1     13.0     33.2     50.1   
3       Charbagh 2026-09-25 20:00:00      1     13.0     33.2     50.1   
4          Chowk 2026-09-25 20:00:00      1     13.0     33.2     50.1   
1    Gomti Nagar 2026-09-25 20:00:00      1     13.0     33.2     50.1   
0     Hazratganj 2026-09-25 20:00:00      1     13.0     33.2     50.1   
5   Indira Nagar 2026-09-25 20:00:00      1     13.0     33.2     50.1   

    rain_24h  rain_72h  elevation  relative_elev